<a href="https://colab.research.google.com/github/SUHANI-21/MachineLearningLab-2/blob/main/Candidate_Elimination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CANDIDATE ELIMINATION ALGORITHM
# Using the same EnjoySport dataset
# ============================================================

import pandas as pd

# ============================================================
# 1. CREATE THE SAME DATASET
# ============================================================

data = {
    'Sky':      ['Sunny', 'Sunny', 'Rainy', 'Sunny', 'Sunny', 'Rainy'],
    'AirTemp':  ['Warm', 'Warm', 'Cold', 'Warm', 'Cold', 'Cold'],
    'Humidity': ['Normal', 'High', 'High', 'High', 'Normal', 'Normal'],
    'Wind':     ['Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Strong'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)

print("Dataset:")
display(df)


Dataset:


,Sky,AirTemp,Humidity,Wind,EnjoySport
0,Sunny,Warm,Normal,Weak,Yes
1,Sunny,Warm,High,Strong,Yes
2,Rainy,Cold,High,Strong,No
3,Sunny,Warm,High,Weak,Yes
4,Sunny,Cold,Normal,Weak,Yes
5,Rainy,Cold,Normal,Strong,No


In [2]:
# ============================================================
# 2. INITIALIZE S AND G
# ============================================================

attributes = list(df.columns[:-1])
n = len(attributes)

# Most specific hypothesis
S = ['ϕ'] * n

# Most general hypothesis
G = [['?'] * n]

print("\nInitial Specific Boundary (S):")
print(S)

print("\nInitial General Boundary (G):")
print(G)




Initial Specific Boundary (S):
['ϕ', 'ϕ', 'ϕ', 'ϕ']

Initial General Boundary (G):
[['?', '?', '?', '?']]


In [3]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def covers(hypothesis, example):
    """
    Check whether a hypothesis covers an example.
    """
    for h, x in zip(hypothesis, example):
        if h != '?' and h != x:
            return False
    return True


def more_general_or_equal(h1, h2):
    """
    Returns True if h1 is more general than or equal to h2.
    """
    for a, b in zip(h1, h2):

        if a == '?':
            continue

        if a == 'ϕ':
            if b != 'ϕ':
                return False

        elif b == 'ϕ':
            continue

        elif a != b:
            return False

    return True


def minimal_generalization(s, example):
    """
    Generalize S enough to cover the positive example.
    """
    new_s = s.copy()

    for i in range(len(s)):

        if s[i] == 'ϕ':
            new_s[i] = example[i]

        elif s[i] != example[i]:
            new_s[i] = '?'

    return new_s


def minimal_specializations(g, example, domains):
    """
    Specialize G so that it no longer covers a negative example.
    """
    specializations = []

    for i in range(len(g)):

        if g[i] == '?':

            for value in domains[i]:

                if value != example[i]:

                    new_g = g.copy()
                    new_g[i] = value
                    specializations.append(new_g)

    return specializations


In [4]:
# ============================================================
# 4. GET ATTRIBUTE DOMAINS
# ============================================================

domains = []

for attribute in attributes:
    domains.append(df[attribute].unique().tolist())

print("\nAttribute Domains:")
for attribute, values in zip(attributes, domains):
    print(attribute, ":", values)


Attribute Domains:
Sky : ['Sunny', 'Rainy']
AirTemp : ['Warm', 'Cold']
Humidity : ['Normal', 'High']
Wind : ['Weak', 'Strong']


In [5]:
# ============================================================
# 5. CANDIDATE ELIMINATION
# ============================================================

for index, row in df.iterrows():

    example = list(row[attributes])
    target = row['EnjoySport']

    print("\n--------------------------------------------")
    print("Example", index + 1, ":", example)
    print("Target:", target)

    # ========================================================
    # POSITIVE EXAMPLE
    # ========================================================

    if target == 'Yes':

        # Remove G hypotheses that don't cover positive example
        G = [
            g for g in G
            if covers(g, example)
        ]

        # Generalize S if required
        if not covers(S, example):

            S = minimal_generalization(S, example)

        # Remove S if it is not more specific than some G
        G = [
            g for g in G
            if more_general_or_equal(g, S)
        ]

    # ========================================================
    # NEGATIVE EXAMPLE
    # ========================================================

    else:

        # Remove S hypotheses that cover negative example
        if covers(S, example):
            print("S covers negative example.")

        # Specialize G hypotheses that cover negative example
        new_G = []

        for g in G:

            if covers(g, example):

                specializations = minimal_specializations(
                    g,
                    example,
                    domains
                )

                for specialization in specializations:

                    if more_general_or_equal(
                        specialization,
                        S
                    ):
                        new_G.append(specialization)

            else:
                new_G.append(g)

        G = new_G

    print("S =", S)
    print("G =", G)


--------------------------------------------
Example 1 : ['Sunny', 'Warm', 'Normal', 'Weak']
Target: Yes
S = ['Sunny', 'Warm', 'Normal', 'Weak']
G = [['?', '?', '?', '?']]

--------------------------------------------
Example 2 : ['Sunny', 'Warm', 'High', 'Strong']
Target: Yes
S = ['Sunny', 'Warm', '?', '?']
G = [['?', '?', '?', '?']]

--------------------------------------------
Example 3 : ['Rainy', 'Cold', 'High', 'Strong']
Target: No
S = ['Sunny', 'Warm', '?', '?']
G = [['Sunny', '?', '?', '?'], ['?', 'Warm', '?', '?']]

--------------------------------------------
Example 4 : ['Sunny', 'Warm', 'High', 'Weak']
Target: Yes
S = ['Sunny', 'Warm', '?', '?']
G = [['Sunny', '?', '?', '?'], ['?', 'Warm', '?', '?']]

--------------------------------------------
Example 5 : ['Sunny', 'Cold', 'Normal', 'Weak']
Target: Yes
S = ['Sunny', '?', '?', '?']
G = [['Sunny', '?', '?', '?']]

--------------------------------------------
Example 6 : ['Rainy', 'Cold', 'Normal', 'Strong']
Target: No
S = 

In [ ]:
# ============================================================
# 6. REMOVE DUPLICATES
# ============================================================

unique_G = []

for g in G:
    if g not in unique_G:
        unique_G.append(g)

G = unique_G


In [6]:
# ============================================================
# 7. FINAL VERSION SPACE
# ============================================================

print("\n============================================")
print("FINAL CANDIDATE ELIMINATION RESULT")
print("============================================")

print("\nMost Specific Hypothesis (S):")
print(S)

print("\nMost General Hypotheses (G):")

for g in G:
    print(g)

print("\n============================================")
print("FINAL VERSION SPACE")
print("============================================")

print("S =", S)
print("G =", G)



FINAL CANDIDATE ELIMINATION RESULT

Most Specific Hypothesis (S):
['Sunny', '?', '?', '?']

Most General Hypotheses (G):
['Sunny', '?', '?', '?']

FINAL VERSION SPACE
S = ['Sunny', '?', '?', '?']
G = [['Sunny', '?', '?', '?']]
